# TC-WPN — Phase 3B: one configuration per session

**Accelerator: GPU T4. Run this FOUR times, changing `CONFIG` each time.
Always via Save Version → Save & Run All (Commit). Never interactively.**

## What went wrong with Phase 3A

Three separate problems, all of them mine.

**1. The job did not fit in a session.** 20 runs at ~48 min is 16 hours against a
12-hour cap. I wrote that number in the notebook and then still wrote a single
cell that attempts all 20 in one pass. It ran until Kaggle killed it.

**2. The output directory reached roughly 8.6 GB.** `train.py` writes a ~440 MB
`best.pt` per run, and nothing deleted them. Twenty checkpoints in
`/kaggle/working` makes committing extremely slow and makes the notebook's
output listing very heavy to render — the most likely reason the page will not
load now.

**3. Interactive sessions lose `/kaggle/working` when they time out.** If Phase
3A was run interactively rather than committed, the outputs were not saved. See
the recovery section at the bottom before assuming the 12 hours are gone.

## What changed here

- **One configuration per session.** 5 seeds × ~48 min ≈ 4 hours, comfortably
  inside the cap with headroom for a slow T4.
- **Checkpoints are deleted after evaluation**, except for the seed used in
  DeLong pairing and blinded evaluation. Output drops from ~8.6 GB to a few
  hundred MB.
- **Subprocess output goes to a log file**, with only the tail printed. The
  notebook JSON stays small enough to open.
- **Fully resumable.** Anything already evaluated is skipped.

In [ ]:
# ===========================================================================
# SET THIS, THEN COMMIT. Run once per value:
#     "aux_only"  ->  "temporal_aux"  ->  "pcw_aux"  ->  "tcwpn_full"
# ===========================================================================
CONFIG = "aux_only"

SEEDS          = [42, 43, 44, 45, 46]
KEEP_CKPT_SEED = 42      # only this seed's best.pt survives, for DeLong + blinding
K              = 5
STEM           = "psych_mimic4idx"
print("this session trains:", CONFIG, "seeds", SEEDS)

In [ ]:
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!pip install -q -r requirements.txt 2>&1 | tail -2

import os
# Quieten the libraries that generate most of the notebook's output volume.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = "src"

import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/test_repo_layout.py", "tests/test_call_arity.py",
                    "-q", "--no-header"],
                   capture_output=True, text=True, env=os.environ)
print(r.stdout[-1500:])
if r.returncode != 0:
    raise SystemExit("Repository layout is broken — fix before spending GPU time.")

In [ ]:
from pathlib import Path
STAGE_A_DS = Path("/kaggle/input/datasets/dulharakaushalya/tc-wpn-stage-a-data")
STAGE_A = next((c for c in (STAGE_A_DS/"data"/"clean", STAGE_A_DS/"clean", STAGE_A_DS)
                if (c/"pkl").exists()), None)
if STAGE_A is None:
    raise SystemExit(f"no pkl/ under {STAGE_A_DS}")

PKL_DIR  = "/kaggle/working/pkl"
PLAN_DIR = str(STAGE_A/"plans")
RESULTS  = "/kaggle/working/results"
LOGS     = "/kaggle/working/logs"
!mkdir -p {PKL_DIR} {LOGS}
!cp {STAGE_A}/pkl/*.pkl {PKL_DIR}/
print("ready")

## Recover anything that already exists

Add every previous result dataset as an input. Predictions files are preferred
over checkpoints: DeLong needs the score vectors, not the model.

In [ ]:
import glob, shutil, os
import pandas as pd

def recover(cfg, seed):
    name = f"{cfg}_k{K}_seed{seed}"
    dst  = f"{RESULTS}/{STEM}/{name}"
    if os.path.exists(f"{dst}/eval_test.json"):
        return "done"
    for target in ("predictions_test.csv", "best.pt"):
        hits = glob.glob(f"/kaggle/input/**/{name}/{target}", recursive=True)
        if hits:
            src = os.path.dirname(sorted(hits)[0])
            os.makedirs(dst, exist_ok=True)
            for f in os.listdir(src):
                p = os.path.join(src, f)
                if os.path.isfile(p):
                    shutil.copy2(p, dst)
            return "recovered_predictions" if target.endswith(".csv") else "recovered_ckpt"
    return "todo"

status = {s: recover(CONFIG, s) for s in SEEDS}
print(pd.Series(status, name="status").to_string())
todo = [s for s, v in status.items() if v == "todo"]
print(f"\nto train: {todo}  (~{len(todo)*48/60:.1f} h)")
if len(todo) * 48 > 11 * 60:
    print("WARNING: this may exceed the 12 h cap. Split SEEDS across two commits.")

## Train, evaluate, then free the disk

Each run's stdout goes to `/kaggle/working/logs/`. Only the last 25 lines print,
which is enough to see the loss trajectory and the locked threshold without
inflating the notebook. The full log is committed with the output if you need it.

In [ ]:
import subprocess, os, time

def run(cmd, logfile, tail=25):
    t0 = time.time()
    with open(logfile, "w") as fh:
        p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, env=os.environ)
    mins = (time.time() - t0) / 60
    lines = open(logfile).read().splitlines()
    print("\n".join(lines[-tail:]))
    print(f"[exit {p.returncode} | {mins:.1f} min | full log: {logfile}]")
    return p.returncode

for seed in SEEDS:
    name = f"{CONFIG}_k{K}_seed{seed}"
    run_dir = f"{RESULTS}/{STEM}/{name}"

    if not os.path.exists(f"{run_dir}/eval_test.json"):
        if not os.path.exists(f"{run_dir}/best.pt"):
            print("=" * 70); print(f"TRAIN {name}"); print("=" * 70)
            rc = run(["python", "-m", "scripts.train",
                      "--config", f"configs/{CONFIG}.yaml",
                      "--k", str(K), "--seed", str(seed), "--stem", STEM,
                      "--pkl-dir", PKL_DIR, "--plan-dir", PLAN_DIR,
                      "--results", RESULTS],
                     f"{LOGS}/train_{name}.log")
            if rc != 0:
                print(f"TRAIN FAILED for {name}; skipping evaluation"); continue

        print("-" * 70); print(f"EVALUATE {name}"); print("-" * 70)
        run(["python", "-m", "scripts.evaluate", "--run", run_dir,
             "--split", "test", "--pkl-dir", PKL_DIR, "--plan-dir", PLAN_DIR,
             "--bootstrap", "2000"],
            f"{LOGS}/eval_{name}.log")

    # Free ~440 MB unless this seed is needed for DeLong pairing / blinding.
    ckpt = f"{run_dir}/best.pt"
    if seed != KEEP_CKPT_SEED and os.path.exists(ckpt):
        os.remove(ckpt)
        print(f"[removed {ckpt} to keep the session output small]")

!du -sh /kaggle/working/results /kaggle/working/pkl /kaggle/working/logs

## Blinded evaluation — only in the session that keeps the checkpoint

In [ ]:
run_dir = f"{RESULTS}/{STEM}/{CONFIG}_k{K}_seed{KEEP_CKPT_SEED}"
if os.path.exists(f"{run_dir}/best.pt"):
    for level in ["anxiety", "anx_meds"]:
        if os.path.exists(f"{run_dir}/eval_test_blind-{level}.json"):
            continue
        print(f"blinded evaluation: {level}")
        run(["python", "-m", "scripts.evaluate", "--run", run_dir,
             "--split", "test", "--blind", level,
             "--pkl-dir", PKL_DIR, "--plan-dir", PLAN_DIR, "--bootstrap", "2000"],
            f"{LOGS}/eval_blind_{level}_{CONFIG}.log")
else:
    print("no checkpoint retained in this session — nothing to blind")

In [ ]:
# This session's numbers. The cross-config aggregation happens in Phase 3C,
# after all four sessions have been committed.
import json, glob
rows = []
for f in sorted(glob.glob(f"{RESULTS}/{STEM}/{CONFIG}_k{K}_seed*/eval_test.json")):
    m = json.load(open(f))["metrics"]
    rows.append({"run": os.path.basename(os.path.dirname(f)),
                 "AUROC": round(m["auroc"], 4),
                 "CI_low": round(m["auroc_ci_lower"], 4),
                 "CI_high": round(m["auroc_ci_upper"], 4),
                 "PR_AUC": round(m["pr_auc"], 4),
                 "F1": round(m["f1_positive"], 4),
                 "Sens": round(m["sensitivity"], 4),
                 "Spec": round(m["specificity"], 4)})
if rows:
    df = pd.DataFrame(rows).set_index("run")
    print(df.to_string())
    print(f"\nAUROC mean {df['AUROC'].mean():.4f}  SD {df['AUROC'].std(ddof=1):.4f}")
    df.to_csv(f"/kaggle/working/{CONFIG}_seed_results.csv")
else:
    print("no completed runs in this session")
print("\nNow: Save Version -> Save & Run All (Commit).")
print("Then change CONFIG and repeat. After all four, run Phase 3C to aggregate.")

## If the Phase 3A page still will not open

The notebook is probably too large to render, not corrupted. Options in order:

1. **Kaggle API from your own machine** — this does not load the page at all:
   ```
   pip install kaggle
   kaggle kernels output dulharakaushalya/tc-wpn-phase-3-five-seeds-the-completed-delon -p ./phase3a_output
   ```
   If Phase 3A was committed, every file it wrote comes down, including any
   `eval_test.json` and `predictions_test.csv` that completed before the cutoff.
   Those are worth recovering — each one is a run you do not have to repeat.

2. **The notebook's Output tab** rather than the editor. It renders the file
   listing without the cell outputs.

3. **Copy & Edit** to fork it. The fork loads a fresh editor; you can then clear
   all outputs and save.

If Phase 3A ran **interactively** rather than as a commit, `/kaggle/working` was
discarded when the session was killed and there is nothing to recover. That is
the case worth checking first, because it determines whether you restart from
zero or from partial results.

Either way, upload whatever you recover as a Kaggle dataset and add it as an
input here — the recovery cell will find it.